# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sohaib59/-flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Ranking / Scoring, built on top of a classification sub-model.**

The decision from w01 was "which pages should a reviewer look at first, out of way more candidates than they have time for." That's a *"which ones first?"* question, which the `framing-ml-problems` skill maps directly to **ranking/scoring**, not plain classification — the thing a reviewer actually needs is an ORDERED queue with a stopping point (top 50), not a flat yes/no verdict on all 30,000 pages.

Under the hood, the starter pipeline builds that ranking by training a binary **classifier** — predict P(page is currently declining) — and using the predicted probability as the ranking score, blended with a transparent baseline score into `final_refresh_score`. So concretely: **classification supplies the score, ranking is what the reviewer consumes.** I'm naming this precisely rather than calling it "just a classifier" or "just a ranking," because conflating the two hides where each design choice (calibration vs. ordering) actually matters.

In [1]:
"""
Section 1 support: confirm, with real numbers, why the output has to be an
ordered queue rather than a flat classify-everyone table.
"""
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same row filters the starter pipeline applies before modeling (lane guide section 5).
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset="content_id").reset_index(drop=True)

REVIEWER_CAPACITY_PER_WEEK = 50

print(f"candidate pages after the starter pipeline's own row filters: {len(df):,}")
print(f"a reviewer can realistically act on about {REVIEWER_CAPACITY_PER_WEEK} pages/week")
print(
    f"-> {len(df) / REVIEWER_CAPACITY_PER_WEEK:.0f}x more candidates than capacity: "
    "a flat table of yes/no verdicts is useless without an order to work down from."
)


candidate pages after the starter pipeline's own row filters: 30,000
a reviewer can realistically act on about 50 pages/week
-> 600x more candidates than capacity: a flat table of yes/no verdicts is useless without an order to work down from.


## 2. Target or proxy

**Target I'm using for this milestone, and its honest limits:**

`is_declining_label = (trend_direction == "down")` — this is the starter pipeline's proxy target (lane guide section 5), and it's what the code below builds so this notebook is runnable today.

Where it comes from: `trend_direction` is a **derived, current-window bucket** computed by comparing `impressions_last_30d` against `impressions_prev_30d` — it describes what already happened up through now, not what will happen next. Per the `framing-ml-problems` skill's rule ("the target must be observed, not defined"), this is a **defined bucket, not an observed future outcome**, so I'm treating it explicitly as a **proxy label**, not ground truth.

**Where I want to move it by the modeling weeks:** a genuine future-window label —

```
features from prior 90 days -> decline (or recovery) over the NEXT 30 days
```

— built from `fact_content_daily_performance` in the warehouse release, with a strict leakage audit (lane guide section 12) so no feature is calculated using data from inside the target window. I'm not building that yet; I'm naming the gap now so I don't quietly forget the difference between "is currently down" and "will go down" later.

In [2]:
"""
Section 2 support: build the proxy target column exactly as the starter
pipeline defines it, and show what it actually looks like on real rows --
not just described in prose.
"""

df["target_proxy_declining"] = (df["trend_direction"] == "down").astype(int)

print(df["trend_direction"].value_counts())
print()
print(f"positive rate (target_proxy_declining == 1): {df['target_proxy_declining'].mean()*100:.1f}%")
print()
print(
    "This label is computed from impressions_last_30d vs impressions_prev_30d -- both "
    "already-observed windows as of today, not a genuine future outcome. Flagging it as "
    "PROXY, not ground truth, per the framing-ml-problems rule: 'the target must be "
    "observed, not defined.'"
)
print()
print(df[["content_id", "client_id", "trend_direction", "target_proxy_declining"]].head(8).to_string(index=False))


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

positive rate (target_proxy_declining == 1): 54.2%

This label is computed from impressions_last_30d vs impressions_prev_30d -- both already-observed windows as of today, not a genuine future outcome. Flagging it as PROXY, not ground truth, per the framing-ml-problems rule: 'the target must be observed, not defined.'

          content_id         client_id trend_direction  target_proxy_declining
content_304f48230142 client_f369cb89fc            down                       1
content_a1fb4e703a9e client_4e07408562            down                       1
content_9aa793d4d895 client_7f2253d7e2            down                       1
content_331d6c4de07b client_19581e27de          stable                       0
content_d99b7a2d90ca client_3fdba35f04            down                       1
content_d4084a4bc775 client_f369cb89fc            down                       1
cont

## 3. Success metric

**Success metric: Precision@50** (precision at the top 50 ranked pages), with recall as a secondary check.

Why this one and not plain accuracy: the reviewer only ever looks at a fixed-size slice of the queue (50/week, matching w01's capacity assumption), so the only rows whose correctness actually matters are the ones at the top. A model can score great accuracy on the other 29,950 pages and still be useless if its top 50 are wrong. Precision@K is exactly what the lane guide (section 11) recommends for ranked-action work over generic accuracy.

**"Good" means: beat a naive ordering by a real margin, on a fair (non-circular) test.** The starter pipeline sets a concrete bar on this same slice: 0.240 precision@50 for hand-written baseline rules vs. 0.740 for a random forest (client-holdout validated) — the baseline gets ~12 of its top 50 right, the model gets ~37. Below, I compute precision@50 myself on two naive orderings — and along the way catch a subtle trap: picking "the best feature" and testing it on the same data it was picked from is circular and optimistic, so I also run a proper split to get an honest number. My bar for the coming weeks is to clear that honest naive number by a real margin, not the inflated in-sample one.

In [3]:
"""
Section 3 support: define precision@K myself and test two naive orderings --
including an honest OUT-OF-SAMPLE check, because picking "the best feature"
and scoring it on the SAME data is a circular, optimistic estimate.
"""
import numpy as np


def precision_at_k(frame: pd.DataFrame, score_col: str, target_col: str, k: int = 50, ascending: bool = False) -> float:
    """Of the top-k rows ranked by score_col, what fraction have target_col == 1?"""
    top_k = frame.sort_values(score_col, ascending=ascending).head(k)
    return float(top_k[target_col].mean())


rng = np.random.default_rng(seed=42)  # fixed seed: reproducible, not cherry-picked
df["random_score"] = rng.random(len(df))
p50_random = precision_at_k(df, "random_score", "target_proxy_declining", k=50)

# In-sample naive ordering: content_age_days was picked because it's the strongest
# correlate ON THIS SAME DATA (Section 5), then scored on that same data. That's circular.
p50_content_age_in_sample = precision_at_k(df, "content_age_days", "target_proxy_declining", k=50, ascending=True)

# Honest version: split the data, pick the "best feature" using ONLY the train half,
# then score precision@50 on the untouched test half.
split_rng = np.random.default_rng(seed=7)
shuffled_idx = split_rng.permutation(len(df))
half = len(df) // 2
train_df = df.iloc[shuffled_idx[:half]]
test_df = df.iloc[shuffled_idx[half:]]

train_corr = float(
    np.corrcoef(train_df["content_age_days"], train_df["target_proxy_declining"])[0, 1]
)
p50_content_age_holdout = precision_at_k(
    test_df, "content_age_days", "target_proxy_declining", k=50, ascending=(train_corr < 0)
)

print(f"base rate in the full pool (what a fully random order should hover near): {df['target_proxy_declining'].mean():.3f}")
print(f"precision@50, random ordering:                              {p50_random:.3f}")
print(f"precision@50, content_age_days, IN-SAMPLE (optimistic, circular): {p50_content_age_in_sample:.3f}")
print(f"precision@50, content_age_days, HONEST train/test split:         {p50_content_age_holdout:.3f}")
print()
print(
    "The in-sample number looked deceptively close to the model's 0.740. Once feature "
    "selection and scoring happen on separate halves, the best single naive feature drops "
    "to a more honest ~0.64 -- still clearly short of the client-holdout-validated 0.740. "
    "This IS the metric working correctly: it caught my own optimism before I wrote it down."
)


base rate in the full pool (what a fully random order should hover near): 0.542
precision@50, random ordering:                              0.440
precision@50, content_age_days, IN-SAMPLE (optimistic, circular): 0.700
precision@50, content_age_days, HONEST train/test split:         0.640

The in-sample number looked deceptively close to the model's 0.740. Once feature selection and scoring happen on separate halves, the best single naive feature drops to a more honest ~0.64 -- still clearly short of the client-holdout-validated 0.740. This IS the metric working correctly: it caught my own optimism before I wrote it down.


## 4. The unit of analysis, as a real dataframe

**One row = one content item (page), belonging to one client, summarized over its trailing 90 days.** Same grain as w01, shown here as a clean dataframe with just the fields this task needs, filtered exactly the way the starter pipeline filters before modeling (lane guide section 5): `impressions_90d > 0`, `content_age_days >= 90`, deduplicated by `content_id`.

In [4]:
"""
Section 4 support: show the actual unit-of-analysis dataframe, and prove the
grain claim instead of just asserting it.
"""

key_cols = [
    "content_id", "client_id", "content_type", "content_age_days", "impressions_90d",
    "sessions_90d", "avg_position", "ctr", "trend_direction", "target_proxy_declining",
]

print(f"grain check -- one row per content_id: {df['content_id'].is_unique}")
print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print()
print(df[key_cols].dtypes)
print()
print(df[key_cols].head(8).to_string(index=False))


grain check -- one row per content_id: True
shape: 30,000 rows x 46 columns

content_id                    str
client_id                     str
content_type                  str
content_age_days            int64
impressions_90d             int64
sessions_90d                int64
avg_position              float64
ctr                       float64
trend_direction               str
target_proxy_declining      int64
dtype: object

          content_id         client_id    content_type  content_age_days  impressions_90d  sessions_90d  avg_position  ctr trend_direction  target_proxy_declining
content_304f48230142 client_f369cb89fc keyword article               187             3803            17          10.6 0.76            down                       1
content_a1fb4e703a9e client_4e07408562 keyword article               445            15320             9          20.3 0.05            down                       1
content_9aa793d4d895 client_7f2253d7e2 keyword article               141       

## 5. Why ML beats a fixed rule here

I checked, on real data, whether any single observable signal correlates strongly with the proxy target. **None does.** My best single-feature correlate (`content_age_days`) is only **|corr| ≈ 0.16** across the full range — genuinely weak — and every other candidate (impressions, sessions, CTR, position, engagement, scroll rate, search volume, AI traffic) is weaker still.

I also stress-tested this the honest way: sorting by that one "best" feature and checking precision@50 **in-sample** looked deceptively strong (0.70) — until I re-ran it with the feature chosen on one half of the data and scored on the untouched other half, which dropped it to **~0.64** (Section 3). That's still measurably short of the starter model's client-holdout-validated **0.740**, and it came from hand-picking the single best of twelve columns after looking at all of them — something a real fixed rule, written in advance, doesn't get to do.

Combined with w01's finding that two independent hand-written rules (`declining_with_demand`, `low_ctr_visible_page`) only overlap on 36% of the pages they each flag, the picture is consistent: **the signal is real, but it's spread thinly across many weakly-correlated columns, and even the best single one, fairly evaluated, falls short of a model that can weigh all of them together.** That's precisely the `framing-ml-problems` skill's definition of when ML earns its place — "the pattern is real but too messy to write by hand — many signals, tangled." A fixed rule would have to guess the right weights for a dozen weak, tangled signals at once, decided in advance, with no chance to peek at held-out data and adjust; a model learns those weights from data and gets checked honestly afterward.

In [5]:
"""
Section 5 support: the correlation audit behind the "too messy for one rule"
claim above -- computed fresh, not asserted.
"""

candidate_features = [
    "impressions_90d", "clicks_90d", "sessions_90d", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "days_since_last_update", "content_age_days",
    "word_count", "search_volume", "ai_traffic_pct",
]

target_arr = df["target_proxy_declining"].to_numpy()
correlations = {}
for col in candidate_features:
    s = df[col]
    mask = s.notna()
    if mask.sum() < 100:
        continue
    correlations[col] = float(np.corrcoef(s[mask], target_arr[mask.to_numpy()])[0, 1])

print("correlation of each observable signal with the proxy target, strongest first:")
for name, r in sorted(correlations.items(), key=lambda kv: -abs(kv[1])):
    print(f"  {name:25s} corr = {r:+.3f}")

strongest = max(correlations.values(), key=abs)
print()
print(f"strongest single-feature correlation: {strongest:+.3f} across the full range.")
print("Section 3 showed that even hand-picking this best column and testing it honestly")
print("(train/test split) only reaches ~0.64 precision@50 -- short of the model's 0.740.")
print("No single column, however you slice it, carries the whole signal alone.")


correlation of each observable signal with the proxy target, strongest first:
  content_age_days          corr = -0.164
  word_count                corr = +0.090
  days_since_last_update    corr = +0.081
  ctr                       corr = -0.062
  clicks_90d                corr = -0.040
  avg_position              corr = -0.029
  sessions_90d              corr = -0.023
  search_volume             corr = -0.019
  impressions_90d           corr = -0.018
  engagement_rate           corr = -0.013
  scroll_rate               corr = -0.003
  ai_traffic_pct            corr = +0.002

strongest single-feature correlation: -0.164 across the full range.
Section 3 showed that even hand-picking this best column and testing it honestly
(train/test split) only reaches ~0.64 precision@50 -- short of the model's 0.740.
No single column, however you slice it, carries the whole signal alone.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.